# emogist: pilot LoRA training

This notebook trains a small **LoRA adapter** on top of Qwen3.5 2B, so it learns to turn a sentence into an emoji sentence.

**Before you start:** in Colab, open **Runtime > Change runtime type**, select **T4 GPU**, and save. Then run the cells from top to bottom, one at a time (Shift+Enter). Run **step 7 only once**.

This first run uses training set A (real sentences). The other pilot runs reuse this notebook with one setting changed.

## 1. Install the libraries

Colab starts as an empty machine, so we install the training tools each time. Versions come from Unsloth's official Qwen3.5 2B notebook, because these libraries change quickly. This takes a few minutes.

If Colab asks you to restart the session afterwards, accept, then continue from step 2.

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision "torchaudio==2.8.0" bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!unsloth install-kernels
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

## 2. Settings

Everything you might change between runs is here.

- **Epochs:** how many times the model sees the whole training set.
- **Batch size:** how many examples it looks at before each small weight update.
- **Learning rate:** how big each update is. Too high and training becomes unstable; too low and it barely learns.
- **LoRA rank:** the size of the add-on. Larger can learn more, but overfits more easily.

In [ ]:
import os
import torch

TRAIN_SET = "A_real"            # A_real, B_ai or C_mixed
BASE_MODEL = "unsloth/Qwen3.5-2B"
RUN_NAME = f"qwen2b_{TRAIN_SET}"

EPOCHS = 3
BATCH_SIZE = 4
LEARNING_RATE = 2e-4
LORA_RANK = 16
MAX_LENGTH = 256                # our examples are short
SEED = 3407

# Short on purpose: the behavior should come from the training examples, not a long prompt.
SYSTEM_PROMPT = "Convert the sentence into a short emoji sentence that keeps its gist. Answer with emojis only."

assert torch.cuda.is_available(), "No GPU: use Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

## 3. Upload the data

Run this cell, then choose these two files from your repo:

- `03_dataset/pilot/train_A_real.jsonl`: the sentences and labels to learn from
- `03_dataset/sentences/real_dev.csv`: 100 real sentences the model never trains on, used to score it afterwards

In [ ]:
from google.colab import files

needed = [f"train_{TRAIN_SET}.jsonl", "real_dev.csv"]
missing = [name for name in needed if not os.path.exists(name)]
if missing:
    print("Please select:", missing)
    files.upload()
missing = [name for name in needed if not os.path.exists(name)]
assert not missing, f"Still missing: {missing}"
print("Data ready:", needed)

## 4. Load the base model

This downloads Qwen3.5 2B. `load_in_16bit` stores the weights with less precision, so the model takes half the GPU memory.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_LENGTH,
    load_in_4bit=False,
    load_in_16bit=True,
    full_finetuning=False,
)
# Qwen3.5 can also read images, so this may be a "processor" wrapping the text tokenizer.
text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)
print("Model loaded")

## 5. Prepare the examples

Each example becomes one text in the exact chat format Qwen expects: instructions, sentence, then the emoji answer. The end-of-message token is included, which teaches the model **when to stop**.

We hold back 10% as a **validation set**. The model doesn't train on it, so its loss tells us whether the model is really learning or just memorizing.

In [ ]:
import json
from datasets import Dataset

def to_messages(sentence, emojis=None):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": sentence}]
    if emojis is not None:
        messages.append({"role": "assistant", "content": emojis})
    return messages

def to_text(row):
    text = tokenizer.apply_chat_template(to_messages(row["sentence"], row["emojis"]),
                                         tokenize=False, enable_thinking=False)
    return {"text": text}

with open(f"train_{TRAIN_SET}.jsonl", encoding="utf-8") as f:
    rows = [json.loads(line) for line in f]

split = Dataset.from_list(rows).train_test_split(test_size=0.1, seed=42)
train_data = split["train"].map(to_text)
validation_data = split["test"].map(to_text)
print(f"{len(train_data)} training examples, {len(validation_data)} validation examples\n")
print(train_data[0]["text"])

## 6. Attach the LoRA adapter

The 2 billion original weights are now frozen. Only the small adapter will train. The printed line shows how few weights that is.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_RANK,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    max_seq_length=MAX_LENGTH,
)
model.print_trainable_parameters()

## 7. Train

The table shows the **training loss** regularly and the **validation loss** after each epoch. Lower is better.

- Both fall: it's learning.
- Training falls while validation rises: it's overfitting.
- Neither moves, or a value becomes `nan`: something is wrong, so stop and send me the output.

In [ ]:
from trl import SFTTrainer, SFTConfig

# Rerunning this cell would keep training the same model and spoil the comparison.
assert "trainer" not in globals(), "Already trained: use Runtime > Restart session, then run from step 2"

new_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_data,
    eval_dataset=validation_data,
    args=SFTConfig(
        dataset_text_field="text",
        max_length=MAX_LENGTH,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        warmup_steps=10,
        lr_scheduler_type="linear",
        weight_decay=0.001,
        optim="adamw_8bit",
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="no",
        seed=SEED,
        output_dir="outputs",
        report_to="none",
    ),
)
result = new_trainer.train()
trainer = new_trainer
print(f"Done in {result.metrics['train_runtime'] / 60:.1f} minutes")

## 8. Try it

A quick look before the real scoring. Feel free to change these sentences.

In [ ]:
assert "trainer" in globals(), "Step 7 has not finished in this session"
FastLanguageModel.for_inference(model)

def to_emojis(sentence):
    prompt = tokenizer.apply_chat_template(to_messages(sentence), tokenize=False,
                                           add_generation_prompt=True, enable_thinking=False)
    inputs = text_tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to("cuda")
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=60, do_sample=False, use_cache=True,
                                eos_token_id=text_tokenizer.eos_token_id,
                                pad_token_id=text_tokenizer.pad_token_id)
    answer = output[0][inputs["input_ids"].shape[1]:]
    return text_tokenizer.decode(answer, skip_special_tokens=True).strip()

for sentence in ["I lost my wallet on the bus this morning.",
                 "Ma sœur m'a dit qu'elle arrivait demain.",
                 "It took me twenty minutes to find my car."]:
    print(sentence, "->", to_emojis(sentence))

## 9. Run it on the dev set

The model labels the 100 dev sentences it has never trained on. The file goes back to your Mac, where we score it with the same round-trip scorer as every other model.

In [ ]:
import csv

with open("real_dev.csv", newline="", encoding="utf-8") as f:
    dev = list(csv.DictReader(f))

for i, row in enumerate(dev, 1):
    row["output"] = to_emojis(row["sentence"])
    if i % 25 == 0:
        print(f"{i}/{len(dev)}")

outputs_file = f"pilot_outputs_{RUN_NAME}.csv"
with open(outputs_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["id", "lang", "sentence", "output"], extrasaction="ignore")
    writer.writeheader()
    writer.writerows(dev)
print("Saved", outputs_file)

## 10. Save your results

Colab deletes everything when the session ends, so **download this zip**. It contains the adapter, the dev outputs and the loss history.

Put it in `04_fine_tuning/runs/` in your repo and tell me when it's there.

In [ ]:
import shutil

run_dir = f"run_{RUN_NAME}"
os.makedirs(run_dir, exist_ok=True)
model.save_pretrained(f"{run_dir}/adapter")
tokenizer.save_pretrained(f"{run_dir}/adapter")
shutil.copy(outputs_file, run_dir)
with open(f"{run_dir}/loss_history.json", "w") as f:
    json.dump(trainer.state.log_history, f, indent=1)
with open(f"{run_dir}/settings.json", "w") as f:
    json.dump({"train_set": TRAIN_SET, "base_model": BASE_MODEL, "epochs": EPOCHS, "batch_size": BATCH_SIZE,
               "learning_rate": LEARNING_RATE, "lora_rank": LORA_RANK, "max_length": MAX_LENGTH,
               "seed": SEED, "system_prompt": SYSTEM_PROMPT}, f, indent=1)

archive = shutil.make_archive(run_dir, "zip", run_dir)
print("Downloading", archive)
files.download(archive)